# Notebook 4 — Full System Overview & Interactive Dashboard

This notebook provides an end-to-end overview of the Cryogenic DR Thermal
Simulation project, combining all analysis modules and showing how to launch
the interactive Plotly Dash dashboard.

## Project structure
```
Project3_CryoThermal/
├── python/
│   ├── cryo_thermal.py      — 6-stage ODE model, cooling power, heat loads
│   ├── run_pipeline.py      — CLI runner: CSV + PNG outputs
│   └── dashboard.py         — Plotly Dash interactive app
├── matlab/
│   ├── cooling_power.m      — Stage cooling power curves
│   ├── heat_loads.m         — Wire + radiation load calculator
│   ├── thermal_model.m      — 6-stage ODE (ode45)
│   ├── run_simulation.m     — MATLAB master script
│   └── CryoMonitorApp.m     — uifigure-based monitoring app
├── notebooks/               — This notebook and analysis notebooks
└── outputs/                 — Generated CSV, PNG outputs
```

## To run the interactive dashboard
```bash
cd Project3_CryoThermal
python python/dashboard.py
# Then open http://127.0.0.1:8050 in your browser
```

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from cryo_thermal import simulate_cooldown, heat_balance, STAGE_NAMES, _G0, _C

print('=== Cryogenic DR Thermal Simulation — System Summary ===')
print()
print('Model: Oxford Instruments Triton 400 (lumped RC, 6-stage)')
print(f'Solver: scipy.integrate.solve_ivp (Radau, stiff)')
print()
print('Stage parameters:')
print(f'{"Stage":<22} {"G [W/K]":>12} {"C [J/K]":>12} {"τ [h]":>10}')
print('─' * 60)
for i in range(1, 6):
    tau_h = _C[i] / _G0[i] / 3600 if _G0[i] > 0 else float('inf')
    print(f'{STAGE_NAMES[i]:<22} {_G0[i]:>12.4g} {_C[i]:>12.4g} {tau_h:>10.2f}')

In [ ]:
# ── Full simulation ───────────────────────────────────────────────────────────
t_h, T = simulate_cooldown(t_hours=10.0, n_eval=3000)
T_ss = T[-1]
balance = heat_balance(T_ss)

print('Steady-state results:')
for i, (nm, t) in enumerate(zip(STAGE_NAMES, T_ss)):
    if t > 1:
        label = f'{t:.2f} K'
    else:
        label = f'{t*1e3:.2f} mK'
    print(f'  {nm:<22}: {label}')

In [ ]:
# ── Summary dashboard figure (6-panel) ───────────────────────────────────────
colors = ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3','#a65628']

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# Panel 1: Full cool-down (log)
ax1 = fig.add_subplot(gs[0, :])
for i in range(6):
    ax1.semilogy(t_h, T[:, i], color=colors[i], lw=2, label=STAGE_NAMES[i])
ax1.set_xlabel('Time (h)', fontsize=11); ax1.set_ylabel('T (K)', fontsize=11)
ax1.set_title('10-Hour Cool-down Transient (all stages)', fontsize=12)
ax1.legend(ncol=3, fontsize=10); ax1.grid(True, which='both', alpha=0.3)

# Panel 2: Sub-K stages zoom (mK)
ax2 = fig.add_subplot(gs[1, 0])
for i in [3, 4, 5]:
    ax2.semilogy(t_h, T[:, i] * 1e3, color=colors[i], lw=2, label=STAGE_NAMES[i])
ax2.axhline(15, ls='--', color='k', lw=1.2, label='15 mK')
ax2.set_xlabel('Time (h)', fontsize=10); ax2.set_ylabel('T (mK)', fontsize=10)
ax2.set_title('Sub-Kelvin Stages', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, which='both', alpha=0.3)

# Panel 3: Heat balance bar
ax3 = fig.add_subplot(gs[1, 1])
stage_labels = [r['stage'] for r in balance]
p_cool = [r['P_cool_W'] * 1e6 for r in balance]
q_in   = [r['Q_in_W']  * 1e6 for r in balance]
x = np.arange(len(stage_labels))
ax3.bar(x - 0.2, p_cool, 0.4, label='P_cool', color='steelblue', alpha=0.8)
ax3.bar(x + 0.2, q_in,   0.4, label='Q_in',   color='tomato',    alpha=0.8)
ax3.set_yscale('log')
ax3.set_xticks(x); ax3.set_xticklabels([s.replace(' Stage','') for s in stage_labels], fontsize=8, rotation=20)
ax3.set_ylabel('Power (µW)', fontsize=10)
ax3.set_title('Heat Balance (log scale)', fontsize=11)
ax3.legend(fontsize=9); ax3.grid(axis='y', alpha=0.3, which='both')

# Panel 4: MXC vs qubit power
ax4 = fig.add_subplot(gs[1, 2])
q_powers = np.logspace(-8, -4, 20)
t_mxc = []
for P_q in q_powers:
    _, Tm = simulate_cooldown(n_eval=500, params={'P_qubit_W': P_q})
    t_mxc.append(Tm[-1, 5] * 1e3)
ax4.semilogx(q_powers * 1e6, t_mxc, 'ko-', ms=5, lw=2)
ax4.axhline(20, color='r', ls='--', lw=1.5, label='20 mK limit')
ax4.set_xlabel('Qubit power (µW)', fontsize=10); ax4.set_ylabel('T_MXC (mK)', fontsize=10)
ax4.set_title('MXC vs. Qubit Load', fontsize=11)
ax4.legend(fontsize=9); ax4.grid(alpha=0.3, which='both')

plt.suptitle('Cryogenic Dilution Refrigerator — Thermal Simulation Summary',
             fontsize=14, fontweight='bold', y=0.99)
plt.savefig('../outputs/full_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/full_summary.png')

In [ ]:
# ── Export final CSV ─────────────────────────────────────────────────────────
import pandas as pd
import os

os.makedirs('../outputs', exist_ok=True)

df = pd.DataFrame({
    'time_h':     t_h,
    **{STAGE_NAMES[i]: T[:, i] for i in range(6)}
})
out_csv = '../outputs/stage_temperatures.csv'
df.to_csv(out_csv, index=False)
print(f'CSV saved: {out_csv}  ({len(df)} rows × {len(df.columns)} columns)')
df.tail(3)

## Launching the interactive Dash dashboard

The full interactive dashboard (`python/dashboard.py`) features:
- **6 temperature gauges** updated in real-time after each simulation run
- **Cool-down curve** with stage selectors
- **Heat balance** bar chart  
- **Interactive sliders** for wire count, emissivity, and qubit power
- **Re-Run button** that triggers a new ODE solve with updated parameters

To launch:
```bash
cd Project3_CryoThermal
python python/dashboard.py
```
Then navigate to **http://127.0.0.1:8050**

In [ ]:
# ── Quick feature validation ──────────────────────────────────────────────────
# Verify expected temperatures are within spec before committing results

checks = [
    ('50K stage',   T_ss[1],      40,  55,   'K'),
    ('4K stage',    T_ss[2],       2,   6,   'K'),
    ('Still',       T_ss[3]*1e3, 200, 900,  'mK'),
    ('Cold plate',  T_ss[4]*1e3,  10,  80,  'mK'),
    ('MXC',         T_ss[5]*1e3,   5,  20,  'mK'),
]

all_pass = True
print('=== Acceptance checks ===')
for name, val, lo, hi, unit in checks:
    ok = lo <= val <= hi
    status = '✓ PASS' if ok else '✗ FAIL'
    if not ok: all_pass = False
    print(f'  {status}  {name:<14}: {val:.2f} {unit}  (expected {lo}–{hi} {unit})')

print()
if all_pass:
    print('All acceptance checks passed. Simulation physics verified.')
else:
    print('WARNING: Some checks failed — review model parameters.')